# 08 — Gold-Schicht und Datenqualität

## Zweck
Aus den Silver-Daten **analysebereite Gold-Tabellen** bauen und einen kurzen Qualitätsbericht
erstellen. Notebook `09` liest danach ausschließlich diese Gold-Tabellen — keine Transformationslogik mehr.

## Methodische Trennung
Historische **EEA-Messdaten** und der **Open-Meteo-Live-Snapshot** bleiben getrennt. Jede Tabelle trägt
`dataset_context` (`eea_historical` oder `open_meteo_live`). Beide dürfen nicht direkt verglichen werden
(Messstation vs. Modellwert).

## Ausgabe (5 Parquet-Dateien in `data/gold/`)
`city_air_quality_daily_summary`, `pollutant_ranking_by_city`, `city_context_air_quality`,
`live_air_quality_latest`, `data_quality_summary`.

## Konfiguration und Silver-Daten laden

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

city_reference = pd.read_parquet(SILVER_DIR / "city_reference.parquet")
city_metadata = pd.read_parquet(SILVER_DIR / "city_metadata.parquet")
eea_daily = pd.read_parquet(SILVER_DIR / "eea_city_daily.parquet")
live_silver = pd.read_parquet(SILVER_DIR / "open_meteo_city_hourly")
print({"eea_daily": len(eea_daily), "live_silver": len(live_silver)})

## Gold 1: Tageswerte je Stadt (eea_historical)
Die Silver-Tageswerte werden um den Stadtnamen ergänzt und einheitlich benannt.

In [ ]:
daily_summary = (
    eea_daily.merge(city_reference[["city_id", "city_name"]], on="city_id", how="left")
    .rename(columns={"mean_value": "avg_value", "observation_count": "measurement_count"})
    .assign(dataset_context="eea_historical")
    [["city_id", "city_name", "date", "pollutant", "avg_value", "min_value", "max_value",
      "measurement_count", "source", "data_status", "dataset_context"]]
)
daily_summary.to_parquet(GOLD_DIR / "city_air_quality_daily_summary.parquet", index=False)
daily_summary.head()

## Gold 2: Schadstoff-Rangfolge je Stadt (eea_historical)
Pro Schadstoff der Stadtmittelwert über den Zeitraum, absteigend gereiht.

In [ ]:
ranking = (
    daily_summary.groupby(["pollutant", "city_id", "city_name"], as_index=False)
    .agg(avg_value=("avg_value", "mean"),
         days=("date", "nunique"),
         measurement_count=("measurement_count", "sum"))
)
ranking["rank"] = ranking.groupby("pollutant")["avg_value"].rank(ascending=False, method="min").astype(int)
ranking["dataset_context"] = "eea_historical"
ranking = ranking.sort_values(["pollutant", "rank"])
ranking.to_parquet(GOLD_DIR / "pollutant_ranking_by_city.parquet", index=False)
ranking.head(10)

## Gold 3: Rangfolge mit Stadtkontext (eea_historical)
Die Rangfolge wird mit den Wikipedia-Metadaten (Bevölkerungsdichte) verknüpft — als **explorativer**
Kontext, nicht als Erklärung.

In [ ]:
context = ranking.merge(
    city_metadata[["city_id", "population", "area_km2", "population_density"]],
    on="city_id", how="left",
)
context.to_parquet(GOLD_DIR / "city_context_air_quality.parquet", index=False)
context.head(10)

## Gold 4: Live-Snapshot je Stadt (open_meteo_live)
Aus dem Spark-Streaming-Output je Stadt der **neueste** Messzeitpunkt.

In [ ]:
live_silver["event_time_ts"] = pd.to_datetime(live_silver["event_time_ts"], utc=True)
latest_idx = live_silver.groupby("city_id")["event_time_ts"].idxmax()
live_latest = (
    live_silver.loc[latest_idx]
    .assign(dataset_context="open_meteo_live")
    [["city_id", "city_name", "event_time_ts", "pm2_5", "pm10", "no2", "dataset_context"]]
    .reset_index(drop=True)
)
live_latest.to_parquet(GOLD_DIR / "live_air_quality_latest.parquet", index=False)
live_latest

## Gold 5: Qualitätsbericht
Zeilenzahlen, fehlende Werte und der abgedeckte historische Zeitraum je Tabelle — die Grundlage,
auf der Notebook `09` seine Aussagen einordnet.

In [ ]:
gold_tables = {
    "city_air_quality_daily_summary": daily_summary,
    "pollutant_ranking_by_city": ranking,
    "city_context_air_quality": context,
    "live_air_quality_latest": live_latest,
}
quality_summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns), "missing_values": int(df.isna().sum().sum())}
    for name, df in gold_tables.items()
])
quality_summary["historical_days"] = daily_summary["date"].nunique()
quality_summary.to_parquet(GOLD_DIR / "data_quality_summary.parquet", index=False)

print({"historische_tage": int(daily_summary["date"].nunique())})
quality_summary

## Nächster Schritt
Notebook `09` ausführen — Analyse, Visualisierung und Ergebnisgeschichte.